# Assignment 4 – Task 2: Evaluating Reweighing (Kamiran & Calders)

**Algorithm:** Reweighing (pre-processing), implementation from IBM's AIF360 library.
**Datasets:** Adult, German Credit and COMPAS.
**Classifier:** Logistic regression (scikit-learn).

## How to run
1. Python 3.9 or newer. Install the packages:
   `pip install aif360 scikit-learn pandas numpy`
2. Run all cells from top to bottom (Jupyter / VS Code / Colab). The second code cell downloads the raw data files into AIF360's data folder automatically, so it needs internet the first time.
3. Settings used: test size = 0.3 (stratified on the label), random seed = 42, `LogisticRegression(max_iter=1000)` on standardised features, consistency with `n_neighbors=5`.

| Dataset | Sensitive attribute | Privileged group (=1) | Unprivileged group (=0) | Favourable label |
|---|---|---|---|---|
| Adult | sex | Male | Female | income > 50K |
| German | age | age > 25 | age ≤ 25 | good credit |
| COMPAS | race | Caucasian | not Caucasian | does not reoffend |

In [3]:
# Uncomment the next line if the packages are not installed yet
%pip install aif360 scikit-learn pandas numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Download the raw data
AIF360 does not download the datasets itself; it expects the original files in its own `data/raw` folder.

In [4]:
import os
import urllib.request
import aif360

raw_folder = os.path.join(os.path.dirname(aif360.__file__), "data", "raw")

files = {
    "adult/adult.data":  "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data",
    "adult/adult.test":  "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test",
    "adult/adult.names": "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.names",
    "german/german.data": "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data",
    "german/german.doc":  "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.doc",
    "compas/compas-scores-two-years.csv": "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv",
}

for name, url in files.items():
    path = os.path.join(raw_folder, name)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if not os.path.exists(path):
        print("Downloading", name)
        urllib.request.urlretrieve(url, path)

print("All data files are in", raw_folder)

All data files are in c:\Users\arman\AppData\Local\Programs\Python\Python313\Lib\site-packages\aif360\data\raw


## 2. Imports and settings

In [5]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score

from aif360.datasets import AdultDataset, GermanDataset, CompasDataset
from aif360.metrics import BinaryLabelDatasetMetric
from aif360.algorithms.preprocessing import Reweighing

SEED = 42
TEST_SIZE = 0.3

pip install 'aif360[AdversarialDebiasing]'
pip install 'aif360[AdversarialDebiasing]'
pip install 'aif360[Reductions]'
pip install 'aif360[Reductions]'
pip install 'aif360[inFairness]'
pip install 'aif360[Reductions]'


## 3. Load the datasets
AIF360 turns each sensitive attribute into 1 (privileged) and 0 (unprivileged). For every dataset we pick one sensitive attribute to study.

In [6]:
datasets = {
    "Adult":  (AdultDataset(),  "sex"),
    "German": (GermanDataset(), "age"),
    "COMPAS": (CompasDataset(), "race"),
}

for name, (data, attr) in datasets.items():
    print(f"{name}: {data.features.shape[0]} rows, {data.features.shape[1]} features, sensitive attribute = {attr}")

Adult: 45222 rows, 98 features, sensitive attribute = sex
German: 1000 rows, 58 features, sensitive attribute = age
COMPAS: 6167 rows, 401 features, sensitive attribute = race


## 4. Fairness measures
- **DP_Diff** (demographic parity difference) = P(favourable | unprivileged) − P(favourable | privileged). 0 means equal; negative means the unprivileged group gets the favourable outcome less often.
- **DIR** (disparate impact ratio) = P(favourable | unprivileged) / P(favourable | privileged). 1 means equal; below 0.8 is the usual warning sign.
- **Consistency** = how similar each person's label is to the labels of their 5 nearest neighbours. 1 means similar people always get the same outcome.

In [7]:
def fairness(data, attr):
    metric = BinaryLabelDatasetMetric(data,
                                      unprivileged_groups=[{attr: 0}],
                                      privileged_groups=[{attr: 1}])
    return {
        "DP_Diff": metric.statistical_parity_difference(),
        "DIR": metric.disparate_impact(),
        "Consistency": float(np.ravel(metric.consistency(n_neighbors=5))[0]),
    }

## 5. Run the full pipeline for each dataset
For each dataset:
1. Measure fairness on the **original labels** (before any mitigation or model).
2. Make a **stratified** train/test split.
3. Fit **Reweighing on the training set only** and get one weight per training row.
4. Train two logistic regressions: a **baseline** without weights and a **reweighed** one with the weights. The sensitive attributes are left out of the model's inputs, as in the paper (assumption A2).
5. Measure fairness and accuracy on the **predictions for the test set**.

In [8]:
results = []

for name, (data, attr) in datasets.items():
    print("=" * 50)
    print(name)

    # Step 1: fairness of the original labels (before mitigation)
    results.append({"Dataset": name, "Stage": "1. Original labels (before)", **fairness(data, attr)})

    # Step 2: stratified train/test split
    row_ids = np.arange(data.features.shape[0])
    train_ids, test_ids = train_test_split(row_ids, test_size=TEST_SIZE,
                                           stratify=data.labels.ravel(), random_state=SEED)
    train = data.subset(train_ids)
    test = data.subset(test_ids)

    # Step 3: Reweighing, fitted on the training data only
    rw = Reweighing(unprivileged_groups=[{attr: 0}], privileged_groups=[{attr: 1}])
    train_rw = rw.fit_transform(train)

    print("Weights per group (same idea as Table 4 in the paper):")
    print(f"  unprivileged + favourable:   {rw.w_up_fav:.3f}")
    print(f"  unprivileged + unfavourable: {rw.w_up_unfav:.3f}")
    print(f"  privileged + favourable:     {rw.w_p_fav:.3f}")
    print(f"  privileged + unfavourable:   {rw.w_p_unfav:.3f}")

    # Check: with the weights, the training data has (almost) no demographic parity gap left
    check = BinaryLabelDatasetMetric(train_rw, unprivileged_groups=[{attr: 0}], privileged_groups=[{attr: 1}])
    print(f"Weighted DP_Diff on training data: {check.statistical_parity_difference():.4f}")

    # Step 4: model inputs without the sensitive attributes, standardised
    keep = [i for i, f in enumerate(data.feature_names) if f not in data.protected_attribute_names]
    scaler = StandardScaler().fit(train.features[:, keep])
    X_train = scaler.transform(train.features[:, keep])
    X_test = scaler.transform(test.features[:, keep])
    y_train = train.labels.ravel()
    y_test = test.labels.ravel()

    models = {
        "2. Baseline LR (no mitigation)": None,                     # all rows count equally
        "3. Reweighed LR (after mitigation)": train_rw.instance_weights,
    }

    for stage, weights in models.items():
        clf = LogisticRegression(max_iter=1000)
        clf.fit(X_train, y_train, sample_weight=weights)
        y_pred = clf.predict(X_test)

        # Step 5: put the predictions in a copy of the test set and measure fairness on them
        test_pred = test.copy(deepcopy=True)
        test_pred.labels = y_pred.reshape(-1, 1)

        results.append({"Dataset": name, "Stage": stage,
                        **fairness(test_pred, attr),
                        "Accuracy": accuracy_score(y_test, y_pred),
                        "Balanced accuracy": balanced_accuracy_score(y_test, y_pred)})

Adult
Weights per group (same idea as Table 4 in the paper):
  unprivileged + favourable:   2.232
  unprivileged + unfavourable: 0.846
  privileged + favourable:     0.790
  privileged + unfavourable:   1.096
Weighted DP_Diff on training data: -0.0000
German
Weights per group (same idea as Table 4 in the paper):
  unprivileged + favourable:   1.173
  unprivileged + unfavourable: 0.744
  privileged + favourable:     0.969
  privileged + unfavourable:   1.080
Weighted DP_Diff on training data: 0.0000
COMPAS
Weights per group (same idea as Table 4 in the paper):
  unprivileged + favourable:   1.059
  unprivileged + unfavourable: 0.937
  privileged + favourable:     0.902
  privileged + unfavourable:   1.149
Weighted DP_Diff on training data: -0.0000


## 6. Results: before vs. after mitigation

In [9]:
results_table = pd.DataFrame(results).round(3)
results_table

,Dataset,Stage,DP_Diff,DIR,Consistency,Accuracy,Balanced accuracy
0,Adult,1. Original labels (before),-0.199,0.363,0.849,NaN,NaN
1,Adult,2. Baseline LR (no mitigation),-0.171,0.328,0.923,0.842,0.755
2,Adult,3. Reweighed LR (after mitigation),-0.084,0.606,0.922,0.840,0.744
3,German,1. Original labels (before),-0.149,0.795,0.681,NaN,NaN
4,German,2. Baseline LR (no mitigation),-0.115,0.852,0.715,0.783,0.715
5,German,3. Reweighed LR (after mitigation),-0.044,0.944,0.747,0.763,0.682
6,COMPAS,1. Original labels (before),-0.097,0.840,0.678,NaN,NaN
7,COMPAS,2. Baseline LR (no mitigation),-0.155,0.785,0.858,0.655,0.645
8,COMPAS,3. Reweighed LR (after mitigation),-0.140,0.805,0.849,0.653,0.643


## 7. How to read the table
- **Row 1** shows how biased the data itself is (computed on the full dataset, original labels).
- **Row 2** shows what a normal model learns from that data (test-set predictions).
- **Row 3** shows the same model after Reweighing (test-set predictions).

Reweighing works if DP_Diff in row 3 is closer to 0 and DIR closer to 1 than in row 2, and the cost is the change in (balanced) accuracy. Consistency is not targeted by Reweighing, which only balances the groups, so it is not expected to improve much.